### 1. Lets simulate MCP server functionality with a simple Python implementation which has the following flow:

```
                 ┌─────────────────┐
                 │      LLM        │
                 │                 │
User ───────────►│ Which tool?     │
                 │                 │
                 └────────┬────────┘
                          │
                 get_weather("Toronto")
                          │
                          ▼
                 ┌─────────────────┐
                 │ Python App      │
                 │                 │
                 │ tool → server   │
                 └────────┬────────┘
                          │
                          ▼
                 ┌─────────────────┐
                 │ Weather MCP     │
                 │ Server          │
                 └────────┬────────┘
                          │
                     weather API
                          │
                          ▼
                    Weather result

```

In [3]:
# -------------------------
# MCP Server 1
# -------------------------

def get_weather(city):
    return f"The weather in {city} is 20°C"


# -------------------------
# MCP Server 2
# -------------------------

def get_customer(customer_id):
    return f"Customer {customer_id}: Nausheen"


# -------------------------
# Tool registry
# -------------------------

tools = {
    "get_weather": {
        "server": "weather_server",
        "function": get_weather,
        "description": "Get weather for a city"
    },

    "get_customer": {
        "server": "database_server",
        "function": get_customer,
        "description": "Get customer information"
    }
}


# -------------------------
# Pretend this came from LLM
# -------------------------

llm_response = {
    "tool": "get_weather",
    "arguments": {
        "city": "Toronto"
    }
}


# -------------------------
# Python routes the call
# -------------------------

tool_name = llm_response["tool"]

tool = tools[tool_name]

print("LLM selected:", tool_name)
print("MCP server:", tool["server"])

result = tool["function"](**llm_response["arguments"])

print("Result:", result)

LLM selected: get_weather
MCP server: weather_server
Result: The weather in Toronto is 20°C


## What actually happens with MCP

With real MCP, instead of:
```
result = tool["function"](**arguments)
```
you would have something conceptually like:
```
result = mcp_client.call_tool(
    server="weather_server",
    tool="get_weather",
    arguments={
        "city": "Toronto"
    }
```

The MCP server owns the implementation of get_weather.

Your LLM application owns the routing/orchestration.

So remember this distinction:

```
                    LLM
                     │
              chooses a TOOL
                     │
                     ▼
              Python Agent
                     │
             finds MCP server
                     │
                     ▼
              MCP Server
                     │
             executes tool
                     │
                     ▼
                 Result
                     │
                     ▼
                    LLM
                     │
               final answer
```

And if the user asks:

"What is the weather in Toronto and what are the orders for customer 123?"

The LLM could produce two tool calls:
```
[
    {
        "tool": "get_weather",
        "arguments": {"city": "Toronto"}
    },
    {
        "tool": "get_orders",
        "arguments": {"customer_id": "123"}
    }
]
```
Your application routes them:
```
get_weather
     ↓
Weather MCP Server

get_orders
     ↓
Database MCP Server
```